# 🧪 W5-D1 概念实验：思维链（CoT）为什么有效？

> 配套阅读：`第5周-Day1-思维链CoT详解.md`（CoT 定义、变体家族、业务案例在那边）
>
> 本 notebook 用可运行的小实验回答四个问题：
> 1. **分步为什么更准？** —— 把难题拆成小步，每步的"单步正确率"如何决定整体正确率
> 2. **Zero-shot / Few-shot CoT 的 prompt 长什么样？自洽性（多链投票）收益如何递减？**
> 3. **CoT 是万能的吗？** —— 简单任务上的"过度思考"反例
> 4. **变体家族在"准确率-成本"平面上怎么选型？**

环境：仅 numpy / 标准库 / matplotlib，全部本地模拟，无网络、无大模型调用。

## 实验 1：分步推理的正确率模型

用一个"能力模型"模拟模型行为：单步正确率随该步难度上升而下降（sigmoid）。
- **直接回答**：一步吞下整个问题，难度 = 总难度 D
- **k 步 CoT**：拆成 k 步，每步难度 ≈ D/k + 每步固定的"写过程开销"难度 ε，全部步骤都对才算对

观察：对足够难的多步题，拆步能显著提升正确率——这就是 CoT 的第一性原理。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def step_acc(difficulty, ability=6.0):
    """单步正确率：难度越高越低。ability 是模型能力（越大越强）。"""
    return 1.0 / (1.0 + np.exp(difficulty - ability))

def task_accuracy(difficulty_total, k_steps, ability=6.0, extra=0.35, n_sim=20000):
    """模拟 n_sim 道题：k 步全部正确才答对。返回正确率。"""
    d_each = difficulty_total / k_steps + extra   # extra = 写出中间步骤的额外负担
    p = step_acc(d_each, ability)
    return (rng.binomial(k_steps, p, n_sim) == k_steps).mean()

D = 9.0          # 一道多步应用题的总难度
print(f"总难度 D={D}，直接回答一步到位 vs 拆成 k 步：")
print(f"{'k步':>4} {'单步难度':>8} {'单步正确率':>10} {'整体正确率':>10}")
for k in [1, 2, 3, 4, 5, 6]:
    d = D / k + 0.35
    print(f"{k:>4} {d:>8.2f} {step_acc(d):>10.1%} {task_accuracy(D, k):>10.1%}")

print()
print("解读：拆步把每步难度降下来，虽然步骤数变多、还有\'写过程\'的开销，")
print("但在多步难题上整体正确率仍然大幅提升 —— 这就是 CoT 的价值来源。")

## 实验 2：两种 CoT Prompt + 自洽性（Self-Consistency）投票

1. 构建 Zero-shot CoT（"让我们一步步思考"）与 Few-shot CoT（带示例）的真实 prompt 文本；
2. 模拟自洽性：单条推理链正确率 p=0.62，采样 k 条独立链**多数投票**，
   用二项分布算出投票正确率——观察收益随 k 递增但**递减**，而成本线性上涨。

In [ ]:
def build_prompt(question, mode="zero_shot", examples=None):
    if mode == "zero_shot":
        return f"问题：{question}\n让我们一步步思考。"
    shots = "\n\n".join(
        f"问题：{q}\n推理：{c}\n答案：{a}" for q, c, a in (examples or []))
    return f"{shots}\n\n问题：{question}\n推理："

examples = [
    ("食堂有23个苹果，用掉20个又买了6个，现在有几个？", "23-20=3；3+6=9", "9"),
    ("一班30人分成6组，每组几人？", "30÷6=5", "5"),
]
q = "一件衬衫原价150元，先打8折再用50元券，最终多少钱？"
print("=== Zero-shot CoT ===\n" + build_prompt(q, "zero_shot"))
print("\n=== Few-shot CoT ===\n" + build_prompt(q, "few_shot", examples))

# --- 自洽性：k 条链多数投票 ---
from math import comb
p_single = 0.62                      # 单链正确率
print(f"\n单链正确率 p={p_single}，多数投票（正确链过半则最终正确，k 取奇数避免平票）：")
print(f"{'k条链':>6} {'投票正确率':>10} {'相对成本':>8} {'较单链提升':>10}")
base = p_single
for k in [1, 3, 5, 7, 9, 13, 21]:
    ks = range(k // 2 + 1, k + 1)    # 需要过半数链正确
    p_vote = sum(comb(k, i) * p_single**i * (1 - p_single)**(k - i) for i in ks)
    gain = f"{p_vote - base:+.1%}" if k > 1 else "  —"
    print(f"{k:>6} {p_vote:>10.1%} {k:>8}x {gain:>10}")

print("\n解读：1→3 条链提升最猛，之后边际收益快速衰减，成本却线性涨——")
print("工程上 k=3~5 通常是性价比区间，别盲目堆链数。")

## 实验 3：CoT 不是万能的 —— 简单任务上的"过度思考"

模拟两类任务：
- **简单事实题**（一步可答）：写推理链反而引入"被中间步骤带偏"的额外错误源
- **多步计算题**：CoT 增益巨大

用两个错误源建模：`跳步错误`（直接答的压缩损失）与 `分心错误`（每多写一步的小概率跑偏）。

In [ ]:
def simulate(n, kind, use_cot):
    """返回 n 道题的答对比例。"""
    if kind == "simple":   # 事实题：CoT 没有分步增益，反而引入'写链跑偏'错误源
        err = 0.13 if use_cot else 0.10
        return 1 - (rng.random(n) < err).mean()
    steps, p_step = (4, 0.93) if use_cot else (1, 0.62)   # 多步题：压缩跳步错误率高
    return (rng.binomial(steps, p_step, n) == steps).mean()

for kind in ["simple", "multi_step"]:
    direct, cot = simulate(4000, kind, False), simulate(4000, kind, True)
    label = "简单事实题" if kind == "simple" else "多步计算题"
    delta = (cot - direct) * 100
    print(f"{label}: 直接回答 {direct:.1%} | CoT {cot:.1%} | 增益 {delta:+.1f} pp")

print("\n解读：简单题上 CoT 增益≈0 甚至为负（多写=多错的机会）；")
print("复杂多步题上 CoT 大幅领先。=> 先判断任务类型，再决定是否上 CoT（对应 md 的误区1）。")

## 实验 4：变体家族的"准确率-成本"选型图

把 md 里的变体家族放到同一平面：x = 相对 token 成本，y = 模拟评测准确率。
橙点 = Pareto 前沿（没有别的点"更便宜且更准"）。

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
rng = np.random.default_rng(7)

variants = {   # (真实准确率, 相对成本/题)
    "直接回答":      (0.55, 1.0),
    "Zero-shot CoT": (0.68, 2.8),
    "Few-shot CoT":  (0.72, 4.0),
    "CoT+自洽(5链)": (0.83, 14.0),
    "ToT(搜索树)":   (0.88, 35.0),
}
n_q = 300
pts = []
for name, (p, cost) in variants.items():
    acc = (rng.random(n_q) < p).mean()   # 一次评测的抽样波动
    pts.append((cost, acc, name))

frontier = [q for q in pts if not any(o is not q and o[0] <= q[0] and o[1] >= q[1] for o in pts)]
front_names = {f[2] for f in frontier}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# 左图：分步 vs 正确率（实验1的结论可视化）
D, ability = 9.0, 6.0
ks = np.arange(1, 7)
accs = []
for k in ks:
    p = 1 / (1 + np.exp(D / k + 0.35 - ability))
    accs.append((rng.binomial(k, p, 10000) == k).mean())
axes[0].plot(ks, np.array(accs) * 100, "o-", color="#219ebc")
axes[0].set_xlabel("拆分步数 k"); axes[0].set_ylabel("整体正确率 (%)")
axes[0].set_title("多步难题：拆步 → 每步更简单 → 整体更准")
axes[0].grid(alpha=0.3)

# 右图：准确率-成本平面 + Pareto 前沿
for cost, acc, name in pts:
    axes[1].scatter(cost, acc * 100, s=90,
                    color="#fb8500" if name in front_names else "#adb5bd")
    axes[1].annotate(name, (cost, acc * 100), fontsize=9,
                     xytext=(6, 4), textcoords="offset points")
fr = sorted(frontier)
axes[1].plot([f[0] for f in fr], [f[1] * 100 for f in fr], "r--", lw=1, alpha=0.6)
axes[1].set_xscale("log")
axes[1].set_xlabel("单题相对成本 (log)"); axes[1].set_ylabel("评测准确率 (%)")
axes[1].set_title("CoT 变体家族：橙=Pareto 前沿")
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Pareto 前沿：", " → ".join(f[2] for f in fr))
print("被支配的点（灰）：只有在前沿方案不可用/预算受限时才考虑。")

## 小结

- **CoT 有效的原因**：拆步降低每步难度，且过程可被校验（自洽性、人工抽检）
- **自洽性收益递减**：k=3~5 是性价比区间
- **不是万能**：简单任务上过度思考得不偿失 → 先分类任务再选策略
- **选型看 Pareto**：准确率与成本一起看（本图与 W5-D7 全景回顾呼应）